# Cusanovich et al. (2018): clustering in t-SNE space, checked by CARVE

Source: Cusanovich et al., "A Single-Cell Atlas of In Vivo Mammalian Chromatin Accessibility." Cell 174.5 (2018): 1309-1324. GEO accession GSE111586.

sci-ATAC-seq of 81,173 nuclei from 13 adult mouse tissues. The source built an LSI (3 percent site filter, TF-IDF, 50-component SVD), ran t-SNE on it (Rtsne, perplexity 30, 5,000 iterations), and clustered the two-dimensional t-SNE with Seurat's graph community detection (Louvain) into 30 clusters.

Clustering a t-SNE can manufacture clusters: the embedding is stochastic and its islands depend on seed and perplexity. This notebook sweeps Louvain resolution, the source's own algorithm, under CARVE with randomized preprocessing (the LSI as-is, or a t-SNE of it at perplexity 30, 45, 60 or 75) and asks which pipeline is stable and generalizable, and where the source's 30-cluster partition sits. The expectation, stated so it can be falsified: clustering in t-SNE space is less stable and less generalizable than clustering the LSI at the same granularity, and 30 clusters is past the point where either criterion starts to fall.

The reference is the source's own 30 clusters. Every comparison with it is reported as agreement, not accuracy.

---

## 0. Configuration

Everything below reads STUDIES["cusanovich"]: the scales, the resolution grid, the resample count and the preprocessing options. The fit needs the graph extra (igraph, for Louvain).

Runtime, estimated before measurement: at SCALE = "publication" (5,000 cells) the fit draws 150 resamples, four fifths of them t-SNE (30 at each perplexity), then runs 15 Louvain configurations. The design estimate, made for 50 t-SNE fits at 1,000 iterations, was 30 to 60 minutes of t-SNE and one to two hours of clustering and random forests on one core, less in parallel; t-SNE now runs 120 fits at the source's 5,000 iterations, and a fit costs more at higher perplexity, so its share can be ten times that or more. SCALE = "dev" (1,500 cells) is for iteration.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from benchmarks._cusanovich_compare import (
    prepare_cusanovich_inputs,
    save_tables,
    selection_summary,
)
from benchmarks._preprocessing import resolve_preprocessing
from benchmarks._studies import (
    STUDIES,
    carve_cache_path,
    fit_or_load_carve,
    load_study,
    study_resolution_grids,
)
from benchmarks.figures import figure_cusanovich_results, figure_reference_scatter
from benchmarks.figures._cusanovich_results import MARKER_SIZE, SOURCE_TSNE_LABELS
from benchmarks.figures._paths import CASE_STUDY_DIR

RANDOM_SEED = 42
SCALE = "dev"  # "dev" or "publication"

# The selection every panel and table reads.
MEASURE = "stability"
RULE = "1se"
NOT_TWO = False

# Figures and tables are written under a scale-qualified directory, so
# development output can never be mistaken for the manuscript's. Only the
# publication directory is ever copied into overleaf/vis/.
OUT_DIR = CASE_STUDY_DIR if SCALE == "publication" else CASE_STUDY_DIR / SCALE
OUT_DIR.mkdir(parents=True, exist_ok=True)

study = STUDIES["cusanovich"]
model_grids = study_resolution_grids(study)
X, y, meta = load_study(study, scale=SCALE)
print(meta["n_cells"], "cells,", meta["n_features"], "LSI components,", meta["scale"], "scale")


def show(fig):
    """Display a figure exactly once, then close it."""
    display(fig)
    plt.close(fig)

---

## 1. The source's partition

The source's clusters on the source's own t-SNE (their Figure 1), restricted to the cells analyzed here. cell_metadata.txt ships both; the loader carries them as y and meta["source_tsne"], row for row with X. The subsample is stratified by cluster, so every cluster is present. Cells whose marker-based label is Unknown stay, because the 30 clusters assign them too.

In [ ]:
unknown = meta["source_labels"]["cell_label"] == "Unknown"
print(y.nunique(), "source clusters; the smallest has", y.value_counts().min(), "cells here")
print(
    f"{unknown.mean():.1%} of the analyzed cells have cell_label Unknown "
    f"({meta['n_cells_full'] - meta['n_cells_annotated']} of {meta['n_cells_full']} in the atlas)"
)

show(
    figure_reference_scatter(
        meta["source_tsne"], y.to_numpy(),
        axis_labels=SOURCE_TSNE_LABELS, title=f"Cusanovich et al.: {y.nunique()} clusters",
        marker_size=MARKER_SIZE, out_dir=OUT_DIR,
        save_name="cusanovich_reference_scatter.png",
    )
)

---

## 2. CARVE fit

Louvain over the study's resolution grid, with one preprocessing pipeline drawn per resample: the LSI as-is, or t-SNE at perplexity 30, 45, 60 or 75 with the source's other settings (5,000 iterations, random initialization); 30 is the source's own perplexity. The LSI itself is computed once, on every cell, as the source computes it; only the second step is refit. Each pipeline is fit separately on each subsample, so an embedding that does not reproduce across fits lowers both criteria, and the generalizability classifier trains on the LSI itself. The cached fit's filename is keyed on the scale, the grid, the resample count and the preprocessing.

In [ ]:
carve = fit_or_load_carve(
    X, y,
    cache_path=carve_cache_path(
        study, scale=SCALE, root=Path("./carve_state_saves"),
        model_grids=model_grids, n_resamples=study.n_resamples,
        preprocessing=study.preprocessing,
    ),
    model_grids=model_grids,
    n_resamples=study.n_resamples,
    randomize_preprocessing=True,
    **resolve_preprocessing(study.preprocessing),
    consensus_anchors=study.consensus_anchors,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
carve.estimator_results_[
    ["resolution", "n_clusters_observed", "ari_stability", "ari_generalizability"]
]

CARVE's own plots of both criteria over the resolution grid: stability in the top row, generalizability in the bottom row. The left column pools every resample. Its dashed line is CARVE's selection under that row's criterion and RULE, so the bottom row marks the generalizability selection, not the MEASURE selection the rest of the notebook reads. The right column splits the same Louvain configuration by pipeline, each line over the resamples that pipeline received. Its dashed line marks the resolution RULE picks among all pipeline rows together, which can differ from the pooled selection. The two panels in a row share a y-axis.

In [ ]:
# Rows: stability, generalizability. Columns: pooled over pipelines, by pipeline.
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey="row")
for row, measure in enumerate(["stability", "generalizability"]):
    carve.plot_metric_over_n_clusters(
        measure=measure, rule=RULE, not_two=NOT_TWO, ax=axes[row, 0],
        title="Pooled over pipelines",
    )
    carve.plot_metric_by_pipeline(
        measure=measure, rule=RULE, not_two=NOT_TWO, ax=axes[row, 1],
        title="By pipeline",
    )
show(fig)

---

## 3. Comparison

Three things set the source's recipe beside CARVE's: the pipeline CARVE rates best at its selected resolution; the source's operating point, the resolution at which clusterings of the t-SNE at the source's perplexity, 30, come nearest the source's cluster count; and the published partition's own generalizability under the classifier CARVE uses, on the same LSI.

In [ ]:
inputs = prepare_cusanovich_inputs(
    X, y.to_numpy(), carve, source_tsne=meta["source_tsne"],
    measure=MEASURE, rule=RULE, not_two=NOT_TWO,
    random_state=RANDOM_SEED, n_jobs=-1,
)
selection_summary(inputs)

In [ ]:
table = carve.preprocessing_results_
table.loc[
    table["method_id"] == inputs.best_pipeline_row["method_id"],
    ["pipeline", "resolution", "n_resamples", "n_clusters_observed",
     "ari_stability", "ari_generalizability"],
]

In [ ]:
# The figure and both tables are written into OUT_DIR. Copying the
# publication figure into overleaf/vis/ stays a manual step.
show(figure_cusanovich_results(inputs, out_dir=OUT_DIR))
save_tables(inputs, OUT_DIR)

---

## 4. Summary

Panel C places the source's recipe on CARVE's pooled curves twice: at the resolution where the t-SNE pipeline reaches the source's cluster count, and as the published partition's generalizability under the same classifier probe. Panel D separates the curves by pipeline, which is where differences between clustering the LSI and clustering its t-SNE, and between perplexities, show. The selection summary, written to cusanovich_selection_summary.csv, holds every number the case study reports, including the agreement (ARI) between CARVE's consensus and the source's clusters; the per-pipeline table is written to cusanovich_preprocessing_results.csv.